# RAG, End to End

This is where [RAG theory](../../docs/08_rag_theory.md) and
[vector DB basics](../03_vector_db) come together into one working pipeline: load a PDF, chunk
it, embed and store the chunks, retrieve the relevant ones for a question, and have an LLM
answer grounded in just those chunks.

**Source document:** `telecom_guide.pdf` — a short reference guide covering mobile networks,
SIM cards, billing, roaming, and account security.

**Status:** this is my first working pass through the hands-on RAG chapter. I'm still partway
through this part of the video, so expect this notebook to get refined as I watch the rest.

```
PDF --load--> pages --chunk--> chunks --embed & store--> Vector DB
                                                               |
                                                     similarity search
                                                               |
question ----------------------embed-------------------------+--> top-k chunks --> prompt --> LLM --> answer
```


## Setup

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

assert os.getenv("GROQ_API_KEY"), "Set GROQ_API_KEY in a .env file first"
print("API key loaded.")

## Step 1 — Load the PDF

`PyPDFLoader` reads every page and returns a list of `Document` objects, each with a
`.page_content` string.

In [ ]:
from langchain_community.document_loaders import PyPDFLoader

PDF_PATH = "telecom_guide.pdf"

loader = PyPDFLoader(PDF_PATH)
pages = loader.load()

print(f"Loaded {len(pages)} pages.")
print(pages[0].page_content[:400])

## Step 2 — Split into chunks

Whole pages are usually too big and too unfocused to embed well. `chunk_size` caps how many
characters go in each chunk; `chunk_overlap` repeats a little text between neighbouring chunks
so a fact sitting right at a chunk boundary doesn't get cut in a way that loses its context.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=600,
    chunk_overlap=100,
    separators=["\n\n", "\n", ".", " "],
)

chunks = splitter.split_documents(pages)
print(f"{len(pages)} pages -> {len(chunks)} chunks")
print(chunks[0].page_content)

## Step 3 — Embed the chunks and store them

Using a small, free, local embedding model (`all-MiniLM-L6-v2`) so this runs without an
embedding API key, and Chroma as the vector store.

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vector_store = Chroma.from_documents(chunks, embeddings)

print(f"Stored {vector_store._collection.count()} chunk vectors.")

## Step 4 — Sanity-check the retriever alone

Before adding the LLM, confirm retrieval alone is pulling back sensible chunks for a question.

In [ ]:
retriever = vector_store.as_retriever(search_kwargs={"k": 3})

test_query = "What is VoLTE?"
for i, doc in enumerate(retriever.invoke(test_query), 1):
    print(f"--- chunk {i} ---")
    print(doc.page_content[:250])
    print()

## Step 5 — Build the full RAG chain

Wiring retriever -> prompt -> LLM together with LangChain's LCEL (`|` operator). The system
prompt explicitly restricts the model to the retrieved context, which is what keeps it from
hallucinating an answer when the guide doesn't actually cover something.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_groq import ChatGroq


def format_docs(docs):
    return "\n\n---\n\n".join(doc.page_content for doc in docs)


SYSTEM_PROMPT = """You are a helpful telecom support assistant.
Answer the question using ONLY the context below. If the context doesn't contain
the answer, say you don't have enough information instead of guessing.

Context:
{context}
"""

prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_PROMPT),
    ("human", "{question}"),
])

llm = ChatGroq(model="qwen/qwen3-32b", temperature=0, reasoning_format="parsed")

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

print("RAG chain ready.")

## Step 6 — Ask it something

In [ ]:
question = "How does international roaming billing work?"
print("Q:", question)
print("A:", rag_chain.invoke(question))

## Step 7 — Inspect retrieval for a harder question

Useful for debugging: see exactly which chunks were handed to the LLM before trusting its
answer.

In [ ]:
debug_question = "What should I do if my SIM card stops being detected?"

docs = retriever.invoke(debug_question)
print(f"Retrieved {len(docs)} chunks for: {debug_question}\n")
for i, doc in enumerate(docs, 1):
    print(f"--- chunk {i} (page {doc.metadata.get('page', '?')}) ---")
    print(doc.page_content)
    print()

print("Answer:", rag_chain.invoke(debug_question))

## What's next for this module

Still refining this — things I expect to add: a persistent (disk-backed) Chroma store instead
of in-memory, an interactive Q&A loop, and maybe a similarity-score threshold so the bot can
say "I don't know" instead of answering from a weak match. Will update
[`docs/00_roadmap.md`](../../docs/00_roadmap.md) once this is fully wired up.